In [ ]:
import torch
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss
import time
import os

DATA_PATH = "/Users/jaygreaser/Documents/GitHub/jsjv-research/data/processed/pubmedqa_filtered.json"
OUTPUT_DIR = "/Users/jaygreaser/Documents/GitHub/jsjv-research/outputs/paired_passes"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(DATA_PATH, "r") as f:
    corpus = json.load(f)

print(f"Corpus loaded: {len(corpus)} samples")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
MODEL_ID = "stanford-crfm/BioMedLM"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32,
    low_cpu_mem_usage=True,
    attn_implementation="eager"
)
model.eval()

n_layers = model.config.n_layer
hidden_size = model.config.n_embd
vocab_size = model.config.vocab_size

print(f"Model loaded: {n_layers} layers, hidden size {hidden_size}")

In [ ]:
documents = []
doc_metadata = []

for sample in corpus:
    for abstract in sample["supporting_abstracts"]:
        documents.append(abstract)
        doc_metadata.append({
            "pubid": sample["pubid"],
            "query": sample["query"],
            "gold_answer": sample["gold_answer"],
            "label": sample["label"]
        })

tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

enc_model = SentenceTransformer("NeuML/pubmedbert-base-embeddings")
doc_embeddings = enc_model.encode(documents, batch_size=32, 
                                   show_progress_bar=True, 
                                   convert_to_numpy=True)
faiss.normalize_L2(doc_embeddings)
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

print(f"Retriever ready. {index.ntotal} documents indexed.")

In [ ]:
def bm25_retrieve(query, k=3):
    scores = bm25.get_scores(query.lower().split())
    top_k = np.argsort(scores)[::-1][:k]
    return [{"abstract": documents[i], "pubid": doc_metadata[i]["pubid"],
             "score": scores[i], "gold_answer": doc_metadata[i]["gold_answer"],
             "label": doc_metadata[i]["label"]} for i in top_k]

def faiss_retrieve(query, k=3):
    qe = enc_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(qe)
    scores, indices = index.search(qe, k)
    return [{"abstract": documents[i], "pubid": doc_metadata[i]["pubid"],
             "score": float(s), "gold_answer": doc_metadata[i]["gold_answer"],
             "label": doc_metadata[i]["label"]} for s, i in zip(scores[0], indices[0])]

def hybrid_retrieve(query, k=3, rrf_k=60):
    bm25_results = bm25_retrieve(query, k=k*2)
    faiss_results = faiss_retrieve(query, k=k*2)
    combined = {}
    for rank, r in enumerate(bm25_results):
        combined[r["abstract"]] = {"meta": r, "score": 1 / (rrf_k + rank + 1)}
    for rank, r in enumerate(faiss_results):
        key = r["abstract"]
        if key in combined:
            combined[key]["score"] += 1 / (rrf_k + rank + 1)
        else:
            combined[key] = {"meta": r, "score": 1 / (rrf_k + rank + 1)}
    return [v["meta"] for v in sorted(combined.values(), 
            key=lambda x: x["score"], reverse=True)[:k]]

def build_rag_prompt(query, k=3):
    results = hybrid_retrieve(query, k=k)
    context_parts = [f"[{i+1}] {r['abstract']}" for i, r in enumerate(results)]
    context = "\n\n".join(context_parts)
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    return prompt, results

def build_suppressed_prompt(query):
    return f"Question: {query}\nAnswer:"

print("Retrieval and prompt functions ready.")

In [ ]:
hook_storage = {
    "hidden_states": {},
    "attention_maps": {},
    "logits_per_layer": {}
}
hooks = []

def make_hidden_state_hook(layer_idx):
    def hook(module, input, output):
        hidden = output[0].detach().squeeze(0)
        hook_storage["hidden_states"][layer_idx] = hidden
        with torch.no_grad():
            normed = model.transformer.ln_f(hidden)
            logits = model.lm_head(normed)
        hook_storage["logits_per_layer"][layer_idx] = logits.detach()
    return hook

for i, block in enumerate(model.transformer.h):
    h = block.register_forward_hook(make_hidden_state_hook(i))
    hooks.append(h)

print(f"Hooks registered on {n_layers} layers.")

In [ ]:
def run_instrumented_pass(prompt, max_length=256):
    for key in hook_storage:
        hook_storage[key].clear()

    inputs = tokenizer(prompt, return_tensors="pt", 
                      truncation=True, max_length=max_length)
    seq_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    for i, attn in enumerate(outputs.attentions):
        hook_storage["attention_maps"][i] = attn.detach().squeeze(0)

    return {
        "hidden_states": {k: v.clone() for k, v in hook_storage["hidden_states"].items()},
        "attention_maps": {k: v.clone() for k, v in hook_storage["attention_maps"].items()},
        "logits_per_layer": {k: v.clone() for k, v in hook_storage["logits_per_layer"].items()},
        "seq_len": seq_len,
        "input_ids": inputs["input_ids"].squeeze(0)
    }

print("Forward pass function ready.")

In [ ]:
VALIDATION_SAMPLES = 20
divergence_log = []

print(f"Running paired forward passes on {VALIDATION_SAMPLES} samples...\n")

for idx, sample in enumerate(corpus[:VALIDATION_SAMPLES]):
    query = sample["query"]
    
    rag_prompt, retrieved = build_rag_prompt(query, k=3)
    suppressed_prompt = build_suppressed_prompt(query)
    
    rag_tensors = run_instrumented_pass(rag_prompt)
    
    sup_tensors = run_instrumented_pass(suppressed_prompt)
    
    rag_final_logits = rag_tensors["logits_per_layer"][n_layers-1][-1]
    sup_final_logits = sup_tensors["logits_per_layer"][n_layers-1][-1]
    
    rag_probs = torch.softmax(rag_final_logits, dim=-1)
    sup_probs = torch.softmax(sup_final_logits, dim=-1)
    
    kl = torch.sum(rag_probs * torch.log((rag_probs + 1e-10) / (sup_probs + 1e-10)))
    
    divergence_log.append({
        "idx": idx,
        "pubid": sample["pubid"],
        "query": query[:60],
        "rag_seq_len": rag_tensors["seq_len"],
        "sup_seq_len": sup_tensors["seq_len"],
        "kl_divergence": float(kl),
        "label": sample["label"]
    })
    
    if (idx + 1) % 5 == 0:
        print(f"  Completed {idx+1}/{VALIDATION_SAMPLES} samples...")

print("\n=== PAIRED PASS DIVERGENCE VALIDATION ===\n")
kl_values = [d["kl_divergence"] for d in divergence_log]
print(f"Mean KL divergence (RAG vs suppressed): {np.mean(kl_values):.4f}")
print(f"Min: {min(kl_values):.4f} | Max: {max(kl_values):.4f}")
print(f"\nAll samples show meaningful divergence: "
      f"{'YES ✓' if np.mean(kl_values) > 0.01 else 'NO — check pipeline'}")
print("\nPer-sample breakdown:")
for d in divergence_log:
    print(f"  [{d['idx']:2d}] pubid={d['pubid']} | "
          f"RAG tokens={d['rag_seq_len']:3d} | "
          f"sup tokens={d['sup_seq_len']:2d} | "
          f"KL={d['kl_divergence']:.4f} | label={d['label']}")

In [ ]:
report_path = os.path.join(OUTPUT_DIR, "sissi_validation_report.json")
with open(report_path, "w") as f:
    json.dump({
        "n_samples": VALIDATION_SAMPLES,
        "mean_kl": float(np.mean(kl_values)),
        "min_kl": float(min(kl_values)),
        "max_kl": float(max(kl_values)),
        "all_diverge": bool(np.mean(kl_values) > 0.01),
        "samples": divergence_log
    }, f, indent=2)

for h in hooks:
    h.remove()

print(f"Validation report saved to {report_path}")